# Notebook 1: RNA-seq Data Preprocessing Pipeline

**From Inference to Prediction** | Bioinformatics Big Data Analysis

---

This notebook demonstrates the complete preprocessing pipeline for RNA-seq gene expression data:

1. **Simulate** a realistic RNA-seq count matrix (negative binomial distribution)
2. **Normalize**: RPKM → TPM → Log2 transformation
3. **Impute** missing values with KNN
4. **Correct** batch effects using empirical Bayes (ComBat)
5. **Visualize** data quality via PCA and t-SNE

### Why These Steps Matter
Raw RNA-seq counts are confounded by:
- **Sequencing depth**: deeper-sequenced samples have higher counts for *every* gene
- **Gene length**: longer genes produce more fragments
- **Batch effects**: technical variation between labs/instruments

Proper preprocessing is the single most important factor for reproducible downstream ML analysis.

In [ ]:
import sys
sys.path.append('..')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

# Set plotting style
plt.rcParams.update({'figure.dpi': 100, 'font.size': 11})
sns.set_style('whitegrid')
np.random.seed(42)

print('Libraries loaded successfully.')

## 1. Simulate Realistic RNA-seq Count Data

RNA-seq read counts follow a **Negative Binomial (NB) distribution** due to biological overdispersion
(variance > mean). This is the model used by DESeq2 and edgeR.

$$K_{ij} \sim \text{NB}(\mu_{ij}, \alpha_i)$$

We simulate:
- **200 samples** across 2 biological conditions (cancer vs. normal)
- **2 batches** representing data from different sequencing runs
- **500 genes** (50 differentially expressed, 450 background)

In [ ]:
def simulate_rnaseq(
    n_samples=200, n_genes=500, n_de_genes=50,
    n_batches=2, fold_change=3.0, dispersion=0.1, seed=42
):
    """Simulate RNA-seq count matrix with batch effects and DE genes."""
    rng = np.random.default_rng(seed)
    
    # Sample metadata
    conditions = np.array(['Cancer'] * (n_samples // 2) + ['Normal'] * (n_samples // 2))
    batches = np.array(['Batch1'] * (n_samples // 2) + ['Batch2'] * (n_samples // 2))
    # Shuffle batch assignment to partially confound with condition
    batch_idx = rng.permutation(n_samples)
    batches[batch_idx[:n_samples//4]] = 'Batch2'
    batches[batch_idx[n_samples//4:n_samples//2]] = 'Batch1'
    
    # Baseline expression (log-normal across genes)
    base_expr = rng.lognormal(mean=6, sigma=2, size=(n_genes, n_samples))
    
    # Add differential expression for DE genes
    de_idx = np.arange(n_de_genes)
    cancer_idx = np.where(conditions == 'Cancer')[0]
    base_expr[de_idx[:, None], cancer_idx] *= fold_change
    
    # Add batch effect (multiplicative, ~2x for batch 2)
    batch2_idx = np.where(batches == 'Batch2')[0]
    batch_effect = rng.uniform(1.5, 2.5, size=(n_genes, 1))
    base_expr[:, batch2_idx] *= batch_effect
    
    # Convert to integer counts (NB-like by adding Poisson noise)
    counts = rng.poisson(base_expr).astype(float)
    
    gene_names = [f'Gene_{i:04d}' for i in range(n_genes)]
    sample_names = [f'Sample_{i:03d}' for i in range(n_samples)]
    
    count_df = pd.DataFrame(counts, index=gene_names, columns=sample_names)
    meta_df = pd.DataFrame({'condition': conditions, 'batch': batches}, index=sample_names)
    
    # Simulate gene lengths (500 - 50000 bp, log-normal)
    lengths = pd.Series(
        rng.lognormal(mean=8, sigma=1.5, size=n_genes).astype(int),
        index=gene_names, name='length_bp'
    )
    
    return count_df, meta_df, lengths


counts, metadata, gene_lengths = simulate_rnaseq()
print(f'Count matrix: {counts.shape} (genes x samples)')
print(f'Metadata:\n{metadata.head()}')
print(f'\nCondition distribution: {metadata.condition.value_counts().to_dict()}')
print(f'Batch distribution:     {metadata.batch.value_counts().to_dict()}')

## 2. Normalization: From Raw Counts to TPM

### RPKM vs TPM
- **RPKM** normalizes for sequencing depth and gene length, but column sums vary → not comparable across samples
- **TPM** fixes this: every sample sums to exactly $10^6$, making it a true relative abundance measure

$$\text{TPM}_i = \frac{C_i / L_{i,\text{kb}}}{\sum_j C_j / L_{j,\text{kb}}} \times 10^6$$

In [ ]:
from src.preprocessing.normalization import rpkm, tpm, log2_transform, z_score_normalize

# Normalize
expr_rpkm = rpkm(counts, gene_lengths)
expr_tpm  = tpm(counts, gene_lengths)
expr_log2 = log2_transform(expr_tpm)

# Verify TPM column sums
print('TPM column sums (should all be 1,000,000):')
print(expr_tpm.sum(axis=0).describe())

# Compare distributions before/after log2 transform
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

sample = 'Sample_000'
axes[0].hist(counts[sample], bins=50, color='steelblue', alpha=0.7, edgecolor='k', lw=0.3)
axes[0].set_title('Raw Counts (highly skewed)', fontweight='bold')
axes[0].set_xlabel('Read Count')

axes[1].hist(expr_tpm[sample], bins=50, color='darkorange', alpha=0.7, edgecolor='k', lw=0.3)
axes[1].set_title('TPM (still skewed)', fontweight='bold')
axes[1].set_xlabel('TPM')

axes[2].hist(expr_log2[sample], bins=50, color='forestgreen', alpha=0.7, edgecolor='k', lw=0.3)
axes[2].set_title('log₂(TPM+1) (approx. Gaussian)', fontweight='bold')
axes[2].set_xlabel('log₂(TPM+1)')

for ax in axes:
    ax.set_ylabel('Frequency')
    ax.grid(alpha=0.3)

plt.suptitle('Effect of Normalization on Count Distribution', fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

## 3. Batch Effect Detection and Correction

PCA is the gold-standard QC tool. Before correction, samples should cluster by **biological condition** (Cancer vs Normal). If they cluster by **batch** instead, downstream analyses will detect batch differences rather than biology.

In [ ]:
from src.features.dimensionality import pca_reduce
from src.preprocessing.batch_correction import combat_correct

# PCA before correction
pc_before, var_before = pca_reduce(expr_log2, n_components=10, return_variance=True)

# Apply ComBat batch correction
expr_corrected = combat_correct(expr_log2, batch=metadata['batch'])

# PCA after correction
pc_after, var_after = pca_reduce(expr_corrected, n_components=10, return_variance=True)

# Plot side by side
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

color_map_cond = {'Cancer': '#d62728', 'Normal': '#1f77b4'}
marker_map_batch = {'Batch1': 'o', 'Batch2': '^'}

for ax, pc, var_ratio, title in [
    (axes[0], pc_before, var_before, 'Before Batch Correction'),
    (axes[1], pc_after, var_after, 'After ComBat Correction'),
]:
    for cond in ['Cancer', 'Normal']:
        for batch in ['Batch1', 'Batch2']:
            mask = (metadata['condition'] == cond) & (metadata['batch'] == batch)
            idx = metadata.index[mask]
            ax.scatter(
                pc.loc[idx, 'PC1'], pc.loc[idx, 'PC2'],
                c=color_map_cond[cond],
                marker=marker_map_batch[batch],
                alpha=0.7, s=50, edgecolors='white', linewidth=0.5,
                label=f'{cond} / {batch}'
            )
    ax.set_xlabel(f'PC1 ({var_ratio[0]:.1%} variance)', fontsize=11)
    ax.set_ylabel(f'PC2 ({var_ratio[1]:.1%} variance)', fontsize=11)
    ax.set_title(title, fontsize=13, fontweight='bold')
    ax.legend(fontsize=8, ncol=2)
    ax.grid(alpha=0.3)

plt.suptitle('PCA: Batch Effect Detection & Correction', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

print('After correction: samples should cluster by CONDITION (color), not batch (shape).')

## 4. t-SNE: Non-linear Structure Discovery

While PCA captures linear structure, **t-SNE** reveals non-linear clusters by minimizing KL divergence between high-dimensional and low-dimensional pairwise similarities. The Student-t kernel in low dimensions prevents the "crowding problem" where all moderate-distance points collapse to the center.

In [ ]:
from src.features.dimensionality import tsne_embed

tsne_coords = tsne_embed(expr_corrected, perplexity=30, n_iter=1000)

fig, ax = plt.subplots(figsize=(8, 6))
for cond, color in color_map_cond.items():
    idx = metadata.index[metadata['condition'] == cond]
    ax.scatter(
        tsne_coords.loc[idx, 'tSNE1'],
        tsne_coords.loc[idx, 'tSNE2'],
        c=color, label=cond, alpha=0.75, s=60, edgecolors='white', linewidth=0.4
    )

ax.set_xlabel('t-SNE 1', fontsize=12)
ax.set_ylabel('t-SNE 2', fontsize=12)
ax.set_title('t-SNE Embedding — Post Batch Correction', fontsize=13, fontweight='bold')
ax.legend(fontsize=11)
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

## Summary

| Step | Input | Output | Purpose |
|------|-------|--------|----------|
| TPM normalization | Raw counts | TPM matrix | Remove sequencing depth & gene length bias |
| log₂(x+1) transform | TPM | Log2-TPM | Variance stabilization, approx. Gaussianity |
| ComBat correction | Log2-TPM + batch labels | Corrected matrix | Remove inter-lab technical variation |
| PCA / t-SNE | Corrected matrix | 2D coordinates | Quality check & biological structure visualization |

The preprocessed `expr_corrected` matrix is now ready for downstream classification and survival analysis.